In [78]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/Motor_Vehicle_Collisions_-_Crashes_20261107.csv", low_memory = False)

In [79]:
df.head()

,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,12/31/2025,21:23,BROOKLYN,11236.0,40.643660,-73.900665,"(40.64366, -73.900665)",ROCKAWAY PKWY,NaN,FLATLANDS AVE,...,Driver Inattention/Distraction,NaN,NaN,NaN,045fa6311c7faedaf93f9ea06a3b8600,Station Wagon/Sport Utility Vehicle,Pick-up Truck,NaN,NaN,NaN
1,12/31/2025,12:15,BRONX,10452.0,40.837140,-73.920480,"(40.83714, -73.92048)",E 168 ST,NaN,RIVER AVE,...,Does Not Apply,NaN,NaN,NaN,f8b21071d7efb59e3f60660c9b29cc96,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN
2,12/31/2025,13:30,NaN,NaN,40.665054,-73.735990,"(40.665054, -73.73599)",NaN,241-42 S CONDUIT AVE,NaN,...,Driver Inattention/Distraction,NaN,NaN,NaN,57bc0d74d20e7ab5ea1687f4ba553787,Sedan,Sedan,NaN,NaN,NaN
3,12/31/2025,05:20,BROOKLYN,11207.0,40.657814,-73.896126,"(40.657814, -73.896126)",LINDEN BLVD,NaN,WILLIAMS AVE,...,Does Not Apply,NaN,NaN,NaN,0ec3dd2412673bec0d26c3ff73f57241,Box Truck,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN
4,12/31/2025,22:03,BRONX,10468.0,40.859978,-73.900790,"(40.859978, -73.90079)",E 184 ST,NaN,MORRIS AVE,...,Failure to Yield Right-of-Way,NaN,NaN,NaN,16f4dc1531345e4b4a06afb822c825fe,Sedan,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN


In [80]:
df.isna().mean().sort_values(ascending=False)

VEHICLE TYPE CODE 5              0.993290
CONTRIBUTING FACTOR VEHICLE 5    0.992987
VEHICLE TYPE CODE 4              0.978105
CONTRIBUTING FACTOR VEHICLE 4    0.976832
VEHICLE TYPE CODE 3              0.918083
CONTRIBUTING FACTOR VEHICLE 3    0.911768
CROSS STREET NAME                0.711163
OFF STREET NAME                  0.470578
VEHICLE TYPE CODE 2              0.338014
ON STREET NAME                   0.288844
ZIP CODE                         0.270233
CONTRIBUTING FACTOR VEHICLE 2    0.238424
BOROUGH                          0.234106
LOCATION                         0.055210
LONGITUDE                        0.055210
LATITUDE                         0.055210
VEHICLE TYPE CODE 1              0.016904
CONTRIBUTING FACTOR VEHICLE 1    0.007200
COLLISION_ID                     0.000000
CRASH DATE                       0.000000
NUMBER OF MOTORIST KILLED        0.000000
NUMBER OF MOTORIST INJURED       0.000000
NUMBER OF CYCLIST KILLED         0.000000
CRASH TIME                       0

In [81]:
# combine date and time into one datetime column
df['crash_dt'] = pd.to_datetime(
    df['CRASH DATE'] + " " + df['CRASH TIME'],
    format = '%m/%d/%Y %H:%M'
)
df['crash_dt'].min(), df['crash_dt'].max()

(Timestamp('2023-01-01 00:00:00'), Timestamp('2025-12-31 23:58:00'))

In [82]:
# Pull out the year, month, day, hour from the date time column
df['year'] = df['crash_dt'].dt.year
df['month'] = df['crash_dt'].dt.month
df['weekday'] = df['crash_dt'].dt.day_name()
df['hour'] = df['crash_dt'].dt.hour

df[['crash_dt', 'year', 'month', 'weekday', 'hour']].head()

,crash_dt,year,month,weekday,hour
0,2025-12-31 21:23:00,2025,12,Wednesday,21
1,2025-12-31 12:15:00,2025,12,Wednesday,12
2,2025-12-31 13:30:00,2025,12,Wednesday,13
3,2025-12-31 05:20:00,2025,12,Wednesday,5
4,2025-12-31 22:03:00,2025,12,Wednesday,22


In [83]:
# check for distributions of time
df['CRASH TIME'].value_counts().head(n=10)

CRASH TIME
00:00    4998
15:00    2564
17:00    2458
16:00    2421
14:00    2276
08:00    2270
18:00    2234
09:00    2184
13:00    2176
12:00    2159
Name: count, dtype: int64

In [84]:
# defining the severity factor
df['injury_crash'] = (df['NUMBER OF PERSONS INJURED'] + df['NUMBER OF PERSONS KILLED']) > 0
df['fatal_crash'] = df['NUMBER OF PERSONS KILLED'] > 0

df[['injury_crash', 'fatal_crash']].agg(['sum', 'mean'])

,injury_crash,fatal_crash
sum,118128.000000,737.000000
mean,0.431939,0.002695


## Recovering missing borough

Borough is missing for 23% of crashes, mostly on highways (40% vs 5% of other crashes),
so dropping those rows would bias the location analysis. Most of them still have
coordinates, so I predict borough from latitude/longitude with k-NN: a crash's nearest
neighbours are almost always in the same borough.

Steps:
1. Keep only coordinates inside NYC. (0, 0) values are placeholders, not real locations.
2. Train k-NN (k=3) on crashes with a known borough, 80/20 train/test split.
3. Check accuracy on the held-out 20%: 99.93%.
4. Predict borough for the 51,130 crashes missing it, stored in a new column
   `borough_filled` so the original `BOROUGH` stays unchanged.

Result: missing borough drops from 23.4% to 4.7%. The rest have no usable coordinates.

In [85]:
# recovering the missing boroughs from known latitude/longitude using K-NN classification
nyc_coords = df['LATITUDE'].between(40.49, 40.92) & df['LONGITUDE'].between(-74.27, -73.68) #coords that are in NYC and not 0,0
borough_present = df['BOROUGH'].notna() # borough not missing


known_data = df[nyc_coords & borough_present] #this is known data
print(len(known_data))
X = known_data[['LATITUDE', 'LONGITUDE']]
y = known_data['BOROUGH']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=0) #80/20 train/test

model = KNeighborsClassifier(n_neighbors=3) #setting up the classifier model with 3 neighbors
model = model.fit(X_train, y_train) #fitting the model for predicting borough

model.score(X_test, y_test) # score on testing data

204502


0.999339869440845

In [86]:
missing_data = df[nyc_coords & ~borough_present]
print(len(missing_data))
#predicting the missing boroughs 
X_missing = missing_data[['LATITUDE', 'LONGITUDE']]
predicted_boroughs = model.predict(X_missing)
predicted_boroughs[:5]

51130


array(['QUEENS', 'QUEENS', 'QUEENS', 'STATEN ISLAND', 'QUEENS'],
      dtype=object)

In [87]:
#putting the predictions back into the data
df['borough_filled'] = df['BOROUGH']

df.loc[nyc_coords & ~borough_present, 'borough_filled'] = predicted_boroughs
df['BOROUGH'].isna().mean(), df['borough_filled'].isna().mean()

(np.float64(0.23410595905412768), np.float64(0.04714735468018122))

In [88]:
df.to_csv("../data/crashes_clean.csv", index = False)